# Notebook 3 — The desk that signs a number


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "text2sql.py").exists():
    raise SystemExit("Set the notebook's working directory to 02_bank_book.")
sys.path.insert(0, str(HERE))

import text2sql as desk
from text2sql import ask

assistant = desk.get_assistant()
assistant.get_graph().print_ascii()


                                                                                  +-----------+......                                                                                 
                                                                                  | __start__ |....................                                                                   
                                                                                 .+-----------+      ............  ..............                                                     
                                                                             ....                                .............   .............                                        
                                                                          ...                                                 .............   ..............                          
                                                                        ..           

## How to read the picture

```text
route            greeting                  -> chat -> END
                 delete / update / insert  -> refuse -> END
                 anything else             -> link

after link       no table                  -> abstain
                 tables found              -> generate -> guard -> execute

after execute    error, or a suspect zero, -> repair once, then guard
                 and a repair is left
                 still broken, or no rows  -> abstain
                 rows                      -> draft

after draft      count, sum, average, or   -> human_review
                 any numeric cell
                 a label, such as a card   -> END
                 type

after human      approve or edit           -> END
                 reject, repair left       -> repair
                 reject, repair spent      -> abstain
```

`human_review` calls `interrupt`. LangGraph runs that function again from the top when the person resumes, and `interrupt` then returns the decision. Do not put work above that call. It would run twice.

Edit replaces the sentence. There is no second model call. The person is the authority.


## Step 1 — Greeting, miss, write, label

**Greeting.** The database stays closed.

**Miss.** HarborDesk leave matches no table. The answer is the fixed abstain sentence.

**Write.** "Delete every loan whose status is D" never becomes SQL.

**Label.** Client 13539's card type is a word. The path does not wait.

The card type leaves without a signature. A signature is for a measured number. `junior` is a label copied from the grid. A count is a measurement, so it waits.


In [2]:
def show(question):
    values, payload = ask(question)
    print("Q:", question)
    for step in values["steps"]:
        print(" ", step)
    if payload:
        print("PAUSED:", payload["reason"])
        print(payload["sql"])
        print(payload["draft"])
    else:
        print(values["answer"])
    print()
    return values, payload

show("Good morning")
show("How many annual leave days does a full-time HarborDesk employee receive?")
show("Delete every loan whose status is D.")
card, card_pause = show("What type of credit card does the client with id 13539 own?")
print("card waited for a person:", card_pause is not None)


Q: Good morning
  chat (database stays closed)
Good morning! How can I help?

Q: How many annual leave days does a full-time HarborDesk employee receive?
  link -> (none)
  abstain
NOT_IN_DATABASE: the Berka bank tables do not answer that question.

Q: Delete every loan whose status is D.
  refuse write
REFUSED: this desk can only read the database. It will not insert, update, or delete.



Q: What type of credit card does the client with id 13539 own?
  link -> card, client, disp
  generate SELECT c.type FROM card AS C JOIN disp AS D ON C.disp_id = D.disp_id WHERE D.client_id = 13539 AND D.type = 'OWNER'
  guard ok
  execute 1 row(s)
  sentence
The client with id 13539 owns a junior credit card.

card waited for a person: False


## Step 2 — The count that waits

Pisek has 46 accounts. `A2` is the district name, and the stored spelling is `Pisek`. The SQL runs on the read-only connection. The sentence waits, because the result is a count.

Before you approve, check the grid. The cell is 46, and the SQL filters `A2 = 'Pisek'`, not `pisek`.


In [3]:
pisek, pause = ask("How many accounts are there in the district of Pisek?")
for step in pisek["steps"]:
    print(" ", step)
print()
print(pause["reason"])
print(pause["sql"])
print(pause["draft"])
print("rows:", pause["rows"])
print("thread:", pisek["thread_id"])


  link -> account, district
  generate SELECT COUNT(*) FROM account JOIN district ON account.district_id = district.district_id WHERE A2 = 'Pisek'
  guard ok
  execute 1 row(s)
  sentence

The result is a measured number. A person signs the sentence before it is sent to the client.
SELECT COUNT(*) FROM account JOIN district ON account.district_id = district.district_id WHERE A2 = 'Pisek'
There are 46 accounts in the district of Pisek.
rows: [[46]]
thread: branch-fb67292b


## Step 3 — The person edits, then the sentence is sent

`resume` takes the same thread id. The action `edit` replaces the sentence with the text the person typed.

Reject would spend the remaining repair, write another SELECT, and either pause again or abstain. The first Pisek pass succeeded, so `repairs_left` is still 1. The first reject therefore tries once more. A second reject finds the repair spent and abstains.


In [4]:
final, again = ask(
    "",
    thread_id=pisek["thread_id"],
    resume={"action": "edit", "text": "CHECKED: Pisek has 46 accounts."},
)
print("paused again:", again is not None)
print("human_action:", final["human_action"])
print(final["answer"])


paused again: False
human_action: edit
CHECKED: Pisek has 46 accounts.


## The same graph, with buttons

```bash
streamlit run app.py
```

| Button | What it does |
|---|---|
| Approve and send | The sentence leaves as written |
| Send the edited text | The path you just ran |
| Reject and try another SQL | One more SELECT, then a new pause or an abstain |

The buttons sit in a form so the edited sentence and the click are sent together.

The checkpointer is in memory. Restarting the app drops a sentence that was waiting. A production desk swaps `MemorySaver` for a sqlite or postgres checkpointer. The nodes do not change.

A warehouse with customer PII would pause before the SQL runs, not only before the sentence is sent. This file is a research extract, and the connection is already read-only, so the pause is on the number the client will read.
